# Two-stage retrieval with Voyage embeddings and reranking on Fireworks AI

Voyage AI's embedding models and rerankers run on Fireworks as dedicated
deployments. This notebook builds a two-stage retrieval pipeline against them:
Voyage embeddings power a MongoDB Atlas `$vectorSearch` recall stage, then a
Voyage reranker reorders the candidates for precision.


In [1]:
!pip install -q requests pymongo

## Prerequisites

You need a [Fireworks API key](https://docs.fireworks.ai/api-reference/create-api-key)
and two dedicated deployments: one Voyage embedding model and one Voyage
reranker. Voyage models are not available on serverless, so
[create a deployment](https://docs.fireworks.ai/api-reference/create-deployment)
for each from the model library, either in the Fireworks UI or via the API.

You also need a MongoDB deployment with Vector Search. Atlas has a
[free tier](https://www.mongodb.com/cloud/atlas/register), or you can use
[MongoDB Community Edition with Vector Search](https://www.mongodb.com/docs/vector-search/tutorials/quick-start/?deployment-type=self&embedding=byo&interface=driver&language=python).


In [2]:
import getpass
import os
import time

import requests

os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Fireworks API key: ")

FIREWORKS_API_BASE = "https://api.fireworks.ai/inference/v1"
HEADERS = {
    "Authorization": f"Bearer {os.environ['FIREWORKS_API_KEY']}",
    "Content-Type": "application/json",
}

# Dedicated deployments are addressed by deployment path, not model name.
# Copy the account and deployment IDs from each deployment's page.

# Note: the numbers below were generated using:
#  Embedder: voyage-4
#  Reranker: voyage-rerank-2-5

ACCOUNT_ID = "<YOUR_ACCOUNT_ID>"
EMBED_DEPLOYMENT_ID = "<YOUR_EMBED_DEPLOYMENT_ID>"
RERANK_DEPLOYMENT_ID = "<YOUR_RERANK_DEPLOYMENT_ID>"

VOYAGE_EMBED_MODEL = f"accounts/{ACCOUNT_ID}/deployments/{EMBED_DEPLOYMENT_ID}"
VOYAGE_RERANK_MODEL = f"accounts/{ACCOUNT_ID}/deployments/{RERANK_DEPLOYMENT_ID}"


def post_with_retry(url, payload, max_retries=8, backoff_seconds=15):
    # On-demand deployments scale to zero when idle, so the first call after
    # a period of inactivity can return 503 while the deployment cold-starts.
    for attempt in range(max_retries):
        response = requests.post(url, json=payload, headers=HEADERS)
        if response.status_code != 503:
            response.raise_for_status()
            return response.json()
        print(f"503 (attempt {attempt + 1}/{max_retries}): {response.text}")
        print(f"Retrying in {backoff_seconds}s...")
        time.sleep(backoff_seconds)
    response.raise_for_status()


## MongoDB setup

Document embeddings are stored in MongoDB, and retrieval runs through its
`$vectorSearch` aggregation stage rather than computing similarity locally.

In [3]:
from pymongo import MongoClient

MONGODB_URI = getpass.getpass("MongoDB connection string: ")

mongo_client = MongoClient(MONGODB_URI)
db = mongo_client["voyage_cookbook"]
collection = db["documents"]


## Embeddings

Voyage models are trained for asymmetric retrieval: pass `input_type="document"`
when embedding your corpus, and `input_type="query"` when embedding a search
query. The model prepends task-specific instructions internally, which improves
retrieval quality over embedding both sides identically.

If you are on Atlas, add an IP access list entry for your client before running
the next cells, or the connection will hang.


In [4]:
def embed(texts, input_type):
    payload = {
        "model": VOYAGE_EMBED_MODEL,
        "input": texts,
        "input_type": input_type,
    }
    data = post_with_retry(f"{FIREWORKS_API_BASE}/embeddings", payload)["data"]
    return [item["embedding"] for item in data]

In [5]:
documents = [
    "John Wick is a 2014 American action thriller film starring Keanu Reeves as a retired hitman.",
    "Ballerina is a 2025 spin-off film set in the John Wick universe, following an assassin trained by the Ruska Roma.",
    "The Ruska Roma is a Romani crime organization featured throughout the John Wick franchise.",
    "Continental Hotels are a network of neutral-ground establishments that serve the assassin underworld in John Wick.",
]

doc_embeddings = embed(documents, input_type="document")
print(f"Embedded {len(doc_embeddings)} documents, dimension = {len(doc_embeddings[0])}")

collection.delete_many({})
collection.insert_many([
    {"text": text, "embedding": embedding}
    for text, embedding in zip(documents, doc_embeddings)
])
print(f"Inserted {len(documents)} documents into MongoDB.")

Embedded 4 documents, dimension = 2048
Inserted 4 documents into MongoDB.


## Create the Atlas Vector Search index

The index must exist and finish building before `$vectorSearch` queries will
work, so this creates it and then blocks until it reports queryable.

In [6]:
INDEX_NAME = "voyage_vector_index"

# Create the index only if it does not already exist.
existing_indexes = [idx["name"] for idx in collection.list_search_indexes()]
if INDEX_NAME not in existing_indexes:
    collection.create_search_index({
        "name": INDEX_NAME,
        "type": "vectorSearch",
        "definition": {
            "fields": [
                {
                    "type": "vector",
                    "path": "embedding",
                    "numDimensions": len(doc_embeddings[0]),
                    "similarity": "cosine",
                }
            ]
        },
    })

print("Waiting for index to become queryable...")
while True:
    idx = next((i for i in collection.list_search_indexes() if i["name"] == INDEX_NAME), None)
    if idx and idx.get("queryable"):
        break
    time.sleep(5)
print("Index ready.")

Waiting for index to become queryable...
Index ready.


## Stage 1: vector similarity search

Embed the query and run MongoDB Atlas's `$vectorSearch` aggregation stage to retrieve the closest documents.

In [7]:
query = "Tell me about the movie Ballerina"
query_embedding = embed([query], input_type="query")[0]

pipeline = [
    {
        "$vectorSearch": {
            "index": INDEX_NAME,
            "path": "embedding",
            "queryVector": query_embedding,
            "numCandidates": 50,
            "limit": 4,
        }
    },
    {
        "$project": {
            "_id": 0,
            "text": 1,
            "score": {"$meta": "vectorSearchScore"},
        }
    },
]

candidates = list(collection.aggregate(pipeline))
for c in candidates:
    print(f"{c['score']:.4f}  {c['text']}")

0.7675  Ballerina is a 2025 spin-off film set in the John Wick universe, following an assassin trained by the Ruska Roma.
0.6125  John Wick is a 2014 American action thriller film starring Keanu Reeves as a retired hitman.
0.5841  The Ruska Roma is a Romani crime organization featured throughout the John Wick franchise.
0.5781  Continental Hotels are a network of neutral-ground establishments that serve the assassin underworld in John Wick.


## Stage 2: rerank

Vector search optimizes for recall over a large corpus. Reranking is a slower,
more accurate pass over the small candidate set, scoring each document against
the query directly.


In [8]:
def rerank(query, documents, top_n=3):
    payload = {
        "model": VOYAGE_RERANK_MODEL,
        "query": query,
        "documents": documents,
        "top_n": top_n,
        "return_documents": True,
    }
    return post_with_retry(f"{FIREWORKS_API_BASE}/rerank", payload)["results"]

candidate_texts = [c["text"] for c in candidates]
reranked = rerank(query, candidate_texts, top_n=3)
for item in reranked:
    print(f"{item['relevance_score']:.6f}  {candidate_texts[item['index']]}")

0.859230  Ballerina is a 2025 spin-off film set in the John Wick universe, following an assassin trained by the Ruska Roma.
0.292920  John Wick is a 2014 American action thriller film starring Keanu Reeves as a retired hitman.
0.287987  Continental Hotels are a network of neutral-ground establishments that serve the assassin underworld in John Wick.
